# Temporal Event Model v2 Diagram

This notebook builds the actual v2 temporal predictor and plots its architecture. It does not require a checkpoint because the diagram is for structure, not trained weights.

The market-structure encoder is intentionally outside the v2 temporal predictor. In training, `train.py` loads the latest v20 encoder, freezes it by default, encodes compact event chunks to `[B, K, 32]`, and sends those embeddings into this temporal model.

In [ ]:
from pathlib import Path
import sys

NOTEBOOK_PATH = Path.cwd()
REPO_ROOT = NOTEBOOK_PATH
for parent in [NOTEBOOK_PATH, *NOTEBOOK_PATH.parents]:
    if (parent / "research" / "temporal_event_model" / "v2" / "model.py").exists():
        REPO_ROOT = parent
        break

if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

print(f"repo_root={REPO_ROOT}")

In [ ]:
import torch
from torch import nn

from research.temporal_event_model.v2.config import DataConfig, ModelConfig
from research.temporal_event_model.v2.model import MarketTemporalReturnPredictor

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
BATCH_SIZE = 2

data_config = DataConfig()
model_config = ModelConfig(
    embedding_dim=32,
    temporal_d_model=256,
    temporal_layers=4,
    temporal_heads=8,
    temporal_ffn_mult=4,
    dropout=0.0,
)
model = MarketTemporalReturnPredictor(
    context_chunks=data_config.context_chunks,
    horizons=data_config.future_event_horizons,
    config=model_config,
).to(DEVICE).eval()

context_embeddings = torch.randn(BATCH_SIZE, data_config.context_chunks, model_config.embedding_dim, device=DEVICE)
class ReturnTensorWrapper(nn.Module):
    def __init__(self, temporal_model: nn.Module) -> None:
        super().__init__()
        self.temporal_model = temporal_model

    def forward(self, context_embeddings: torch.Tensor) -> torch.Tensor:
        return self.temporal_model.predict_return_tensor(context_embeddings)


diagram_model = ReturnTensorWrapper(model).to(DEVICE).eval()

with torch.no_grad():
    output = model(context_embeddings)

print(f"context_embeddings: {tuple(context_embeddings.shape)}")
print(f"return_prediction_norm: {tuple(output.return_prediction_norm.shape)}")
print(f"horizons: {data_config.future_event_horizons}")

## Torchinfo Summary

In [ ]:
try:
    from torchinfo import summary
    summary_result = summary(
        diagram_model,
        input_data=(context_embeddings,),
        depth=5,
        col_names=("input_size", "output_size", "num_params", "trainable"),
        verbose=0,
    )
    summary_result
except ModuleNotFoundError:
    print("torchinfo is not installed in this environment.")


## Torchview Diagram

In [ ]:
diagram_path = REPO_ROOT / "research" / "temporal_event_model" / "v2" / "artifacts" / "v2_temporal_predictor_graph"
diagram_path.parent.mkdir(parents=True, exist_ok=True)

import os
import shutil


def add_graphviz_to_path() -> str | None:
    existing = shutil.which("dot")
    if existing:
        return existing
    candidates = []
    conda_prefix = Path(sys.executable).resolve().parents[1]
    candidates.append(conda_prefix / "Library" / "bin" / "dot.exe")
    for env_key in ("ProgramFiles", "ProgramFiles(x86)"):
        root = os.environ.get(env_key)
        if root:
            candidates.extend(Path(root).glob("Graphviz*/bin/dot.exe"))
            candidates.extend((Path(root) / "Graphviz" / "bin").glob("dot.exe"))
    for candidate in candidates:
        if candidate.exists():
            os.environ["PATH"] = str(candidate.parent) + os.pathsep + os.environ.get("PATH", "")
            return str(candidate)
    return None


def draw_fx_fallback(model: nn.Module, output_path: Path):
    import matplotlib
    matplotlib.use("Agg", force=True)
    import matplotlib.pyplot as plt
    import torch.fx

    traced = torch.fx.symbolic_trace(model.cpu().eval())
    nodes = [node for node in traced.graph.nodes]
    names = []
    for node in nodes:
        if node.op == "call_module":
            module = traced.get_submodule(str(node.target))
            label = f"{node.target}\n{module.__class__.__name__}"
        elif node.op == "placeholder":
            label = "context_embeddings\n[B, K, 32]"
        elif node.op == "output":
            label = "return_prediction_norm\n[B, 8]"
        else:
            label = f"{node.op}\n{node.target}"
        names.append(label)

    fig_height = max(8, 0.55 * len(names))
    fig, ax = plt.subplots(figsize=(12, fig_height))
    ax.set_axis_off()
    y_values = list(reversed(range(len(names))))
    for idx, (label, y) in enumerate(zip(names, y_values)):
        ax.text(
            0.5,
            y,
            label,
            ha="center",
            va="center",
            bbox={"boxstyle": "round,pad=0.35", "facecolor": "#f7fbff", "edgecolor": "#1f77b4"},
            fontsize=9,
        )
        if idx < len(names) - 1:
            ax.annotate("", xy=(0.5, y - 0.38), xytext=(0.5, y - 0.72), arrowprops={"arrowstyle": "->", "lw": 1.2})
    ax.set_ylim(-1, len(names))
    ax.set_xlim(0, 1)
    fig.tight_layout()
    fig.savefig(output_path, dpi=180, bbox_inches="tight")
    return fig


dot_path = add_graphviz_to_path()
if dot_path:
    print(f"Graphviz dot found: {dot_path}")
else:
    print("Graphviz dot was not found on PATH; using torch.fx + matplotlib fallback diagram.")

try:
    from torchview import draw_graph
    if not dot_path:
        raise RuntimeError("torchview requires Graphviz dot; using fallback")
    graph = draw_graph(
        diagram_model,
        input_data=(context_embeddings,),
        expand_nested=True,
        depth=5,
        device=str(DEVICE),
        graph_name="v2_temporal_return_predictor",
        save_graph=True,
        filename=str(diagram_path),
    )
    graph.visual_graph
except ModuleNotFoundError:
    print("torchview is not installed; using torch.fx + matplotlib fallback diagram.")
    draw_fx_fallback(diagram_model, diagram_path.with_suffix(".fx.png"))
except Exception as exc:
    print(f"torchview graph failed: {exc!r}")
    draw_fx_fallback(diagram_model, diagram_path.with_suffix(".fx.png"))


## Optional: Full Training Shape Contract

The full training path is:

1. `context_header_uint8`: `[B, K, 14]`
2. `context_events_uint8`: `[B, K, 128, 16]`
3. v20 market encoder, called over flattened `[B*K, 14]` and `[B*K, 128, 16]`
4. `context_embeddings`: `[B, K, 32]`
5. v2 temporal model output: `[B, 8]` normalized return predictions for event horizons `(8, 16, 32, 64, 128, 256, 512, 1024)`.